In [15]:
# %%
import pandas as pd
from sqlalchemy import create_engine, text
import urllib.parse

# =========================
# 0) DB 설정
# =========================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA_NAME = "d1_machine_log"
TABLE_NAME  = "Main_machine_log"

def get_engine(config=DB_CONFIG):
    user = config["user"]
    password = urllib.parse.quote_plus(config["password"])
    host = config["host"]
    port = config["port"]
    dbname = config["dbname"]
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(url)

engine = get_engine()

# =========================
# 1) 데이터 로딩 (✅ 모든 end_day)
# =========================
sql = text(f"""
SELECT end_day, time, contents
FROM {SCHEMA_NAME}."{TABLE_NAME}"
ORDER BY end_day, time
""")

df = pd.read_sql_query(sql, engine)

print("[INFO] 원본 shape:", df.shape)
print("[INFO] end_day unique count:", df["end_day"].nunique())
display(df.head())

# =========================
# 2) time/ts 생성
# =========================
df["time_td"] = pd.to_timedelta(df["time"].astype(str), errors="coerce")
df["end_day_str"] = df["end_day"].astype(str)
df["ts"] = pd.to_datetime(df["end_day_str"] + " " + df["time"].astype(str), errors="coerce")

df = df.dropna(subset=["time_td", "ts"]).sort_values(["end_day", "time_td"]).reset_index(drop=True)

def format_time(td):
    if pd.isna(td):
        return None
    total = td.total_seconds()
    h = int(total // 3600)
    m = int((total % 3600) // 60)
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

def ts_to_time_str(ts: pd.Timestamp) -> str:
    # "HH:MM:SS.ss" (소수 둘째)
    return f"{ts.hour:02d}:{ts.minute:02d}:{ts.second:02d}.{int(ts.microsecond/10000):02d}"

# =========================
# 3) 군집 분리(기존 유지)
# =========================
pattern_cluster1 = r"UP1|FCT1|FCT2|VISION1|UP2"
pattern_cluster2 = r"UP4|FCT3|FCT4|VISION2|UP5"

df_c1 = df[df["contents"].str.contains(pattern_cluster1, regex=True, na=False)].copy()
df_c2 = df[df["contents"].str.contains(pattern_cluster2, regex=True, na=False)].copy()

df_c1 = df_c1.sort_values(["end_day", "time_td"]).reset_index(drop=True)
df_c2 = df_c2.sort_values(["end_day", "time_td"]).reset_index(drop=True)

print("[INFO] 군집1 shape:", df_c1.shape)
print("[INFO] 군집2 shape:", df_c2.shape)

# =========================
# 4) 핵심: "미사용 시작 ~ 사용 복귀 완료" 구간 생성
#    - from_time: 미사용 이벤트(둘 중 먼저)
#    - to_time  : 사용 이벤트가 둘 다 발생한 시점(마지막)
# =========================
def build_disable_enable_intervals(df_cluster, station_name,
                                  disable_set, enable_set,
                                  input_phrase,
                                  topn=5):
    """
    end_day 별로 interval 생성:
      - disable_set 중 하나 발생 -> from_ts(시작)
      - 이후 enable_set이 모두 관측되는 시점까지 진행
      - to_ts = 관측된 enable 시각 중 가장 늦은 값
    interval(from~to) 내 input_phrase 건수 계산 후 TopN 반환
    """
    interval_rows = []

    for day, g in df_cluster.groupby("end_day", sort=False):
        g = g.sort_values("time_td").reset_index(drop=True)

        # input ts 미리 추출(카운트용)
        g_in = g[g["contents"].str.contains(input_phrase, regex=False, na=False)][["ts"]].copy()

        i = 0
        while i < len(g):
            c = g.loc[i, "contents"]

            # disable 이벤트 시작점
            if c in disable_set:
                from_ts = g.loc[i, "ts"]

                seen_enable = {k: None for k in enable_set}

                j = i + 1
                while j < len(g):
                    cj = g.loc[j, "contents"]
                    if cj in enable_set and seen_enable[cj] is None:
                        seen_enable[cj] = g.loc[j, "ts"]

                        # enable_set 모두 관측되면 종료
                        if all(v is not None for v in seen_enable.values()):
                            to_ts = max(seen_enable.values())  # 마지막 enable 시각

                            # interval 내 투입 카운트
                            if g_in.empty:
                                cnt = 0
                            else:
                                cnt = int(((g_in["ts"] >= from_ts) & (g_in["ts"] <= to_ts)).sum())

                            interval_rows.append({
                                "end_day": day,
                                "station": station_name,
                                "from_ts": from_ts,
                                "to_ts": to_ts,
                                "from_time": ts_to_time_str(from_ts),
                                "to_time": ts_to_time_str(to_ts),
                                "input_count": cnt,
                            })

                            # 중복/중첩 방지: 현재 enable 완료 지점 이후로 이동
                            i = j
                            break
                    j += 1

            i += 1

    intervals = pd.DataFrame(interval_rows)
    if intervals.empty:
        return pd.DataFrame(columns=["end_day","station","from_time","to_time","input_count","from_ts","to_ts"])

    # TopN: 투입 개수 기준 (전체 end_day 통합해서 TopN)
    intervals = intervals.sort_values(["input_count","end_day","from_ts"], ascending=[False, True, True]).reset_index(drop=True)
    return intervals.head(topn)

# =========================
# 5) interval별 투입행 CT 생성
# =========================
def build_ct_from_intervals(df_cluster, intervals_df, input_phrase):
    """
    intervals_df 각 구간(from_ts~to_ts)에서 input_phrase 행만 추출하여
    ct = 현재 - 이전(초) 계산 (첫 행 NaN)
    """
    out_list = []
    if intervals_df.empty:
        return pd.DataFrame(columns=["end_day","station","from_time","to_time","contents","time","ct"])

    for _, r in intervals_df.iterrows():
        day = r["end_day"]
        from_ts = r["from_ts"]
        to_ts   = r["to_ts"]

        g = df_cluster[df_cluster["end_day"] == day].copy()
        g = g[(g["ts"] >= from_ts) & (g["ts"] <= to_ts)].copy()
        g = g[g["contents"].str.contains(input_phrase, regex=False, na=False)].copy()

        if g.empty:
            continue

        g = g.sort_values("time_td").reset_index(drop=True)
        g["ct"] = g["time_td"].diff().dt.total_seconds().round(2)  # 현재-이전, 첫 행 NaN
        g["time"] = g["time_td"].apply(format_time)

        out_list.append(pd.DataFrame({
            "end_day": g["end_day"],
            "station": r["station"],
            "from_time": r["from_time"],
            "to_time": r["to_time"],
            "contents": g["contents"],
            "time": g["time"],
            "ct": g["ct"],
        }))

    if not out_list:
        return pd.DataFrame(columns=["end_day","station","from_time","to_time","contents","time","ct"])
    return pd.concat(out_list, ignore_index=True)

# =========================
# 6) Boxplot 대표값 DF
# =========================
def boxplot_summary(ct_df, station_name):
    x = ct_df["ct"].dropna().astype(float) if not ct_df.empty else pd.Series([], dtype=float)

    if x.empty:
        return {
            "station": station_name,
            "minimum": pd.NA, "q1": pd.NA, "median": pd.NA, "q3": pd.NA,
            "maximum": pd.NA, "upper_outlier": pd.NA, "final_ct": pd.NA
        }

    q1 = x.quantile(0.25)
    med = x.quantile(0.50)
    q3 = x.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr  # upper_outlier

    x_in = x[(x >= lower) & (x <= upper)]
    if x_in.empty:
        return {
            "station": station_name,
            "minimum": pd.NA, "q1": round(float(q1),2), "median": round(float(med),2), "q3": round(float(q3),2),
            "maximum": pd.NA, "upper_outlier": round(float(upper),2), "final_ct": pd.NA
        }

    minimum = float(x_in.min())
    maximum = float(x_in.max())
    final_ct = float(x[x <= upper].mean())  # upper_outlier 초과 제외 평균

    return {
        "station": station_name,
        "minimum": round(minimum, 2),
        "q1": round(float(q1), 2),
        "median": round(float(med), 2),
        "q3": round(float(q3), 2),
        "maximum": round(maximum, 2),
        "upper_outlier": round(float(upper), 2),
        "final_ct": round(final_ct, 2),
    }

# =========================================================
# 군집1 (Vision1): FCT1/2 기준
# =========================================================
disable_c1 = {"FCT1 미사용으로 변경", "FCT2 미사용으로 변경"}
enable_c1  = {"FCT1 사용으로 변경", "FCT2 사용으로 변경"}
input_c1   = "PLC: VISION1 : 투입"

top5_c1 = build_disable_enable_intervals(df_c1, "Vision1", disable_c1, enable_c1, input_c1, topn=5)

print("====================================================")
print("[군집1/Vision1] Top5 interval (from_time~to_time) by input_count (전체 end_day 통합)")
print("====================================================")
display(top5_c1[["end_day","station","from_time","to_time","input_count"]])

df_c1_ct = build_ct_from_intervals(df_c1, top5_c1, input_c1)

print("====================================================")
print("[군집1/Vision1] Top5 interval 내 'PLC: VISION1 : 투입' CT 전체")
print("====================================================")
display(df_c1_ct)

# =========================================================
# 군집2 (Vision2): FCT3/4 기준
# =========================================================
disable_c2 = {"FCT3 미사용으로 변경", "FCT4 미사용으로 변경"}
enable_c2  = {"FCT3 사용으로 변경", "FCT4 사용으로 변경"}
input_c2   = "PLC: VISION2 : 투입"

top5_c2 = build_disable_enable_intervals(df_c2, "Vision2", disable_c2, enable_c2, input_c2, topn=5)

print("====================================================")
print("[군집2/Vision2] Top5 interval (from_time~to_time) by input_count (전체 end_day 통합)")
print("====================================================")
display(top5_c2[["end_day","station","from_time","to_time","input_count"]])

df_c2_ct = build_ct_from_intervals(df_c2, top5_c2, input_c2)

print("====================================================")
print("[군집2/Vision2] Top5 interval 내 'PLC: VISION2 : 투입' CT 전체")
print("====================================================")
display(df_c2_ct)

# =========================================================
# Boxplot 대표값 요약
# =========================================================
df_box_summary = pd.DataFrame([
    boxplot_summary(df_c1_ct, "Vision1"),
    boxplot_summary(df_c2_ct, "Vision2"),
], columns=["station","minimum","q1","median","q3","maximum","upper_outlier","final_ct"])

print("====================================================")
print("[Boxplot 대표값] final_ct = upper_outlier 제외 평균")
print("====================================================")
display(df_box_summary)

[INFO] 원본 shape: (2641752, 3)
[INFO] end_day unique count: 39


,end_day,time,contents
0,20250930,01:18:43.73,PLC: STF : 트레이 투입
1,20250930,01:18:43.74,PLC: 좌측 START
2,20250930,01:18:49.72,PLC: 우측 START
3,20250930,01:18:58.19,PLC: UP5 : 배출
4,20250930,01:18:58.19,PLC: TE3 : 투입


[INFO] 군집1 shape: (1083295, 6)
[INFO] 군집2 shape: (859850, 6)
[군집1/Vision1] Top5 interval (from_time~to_time) by input_count (전체 end_day 통합)


,end_day,station,from_time,to_time,input_count
0,20251112,Vision1,08:34:14.93,17:43:11.56,1346
1,20251117,Vision1,11:53:09.86,20:02:17.64,1264
2,20251013,Vision1,08:34:44.11,12:18:19.75,775
3,20251106,Vision1,14:49:01.50,17:16:27.33,300
4,20251031,Vision1,20:28:52.57,22:44:54.98,184


[군집1/Vision1] Top5 interval 내 'PLC: VISION1 : 투입' CT 전체


...

,end_day,station,from_time,to_time,input_count
0,20251015,Vision2,08:24:39.00,13:55:29.92,900
1,20251031,Vision2,20:26:37.32,22:45:07.38,252
2,20251030,Vision2,20:25:58.41,22:24:34.27,84
3,20251016,Vision2,08:51:06.78,09:35:14.92,68
4,20251010,Vision2,18:06:35.67,18:19:57.76,2


[군집2/Vision2] Top5 interval 내 'PLC: VISION2 : 투입' CT 전체


,end_day,station,from_time,to_time,contents,time,ct
0,20251015,Vision2,08:24:39.00,13:55:29.92,PLC: VISION2 : 투입,08:26:00.68,NaN
1,20251015,Vision2,08:24:39.00,13:55:29.92,PLC: VISION2 : 투입,08:30:45.99,285.31
2,20251015,Vision2,08:24:39.00,13:55:29.92,PLC: VISION2 : 투입,08:31:00.04,14.05
3,20251015,Vision2,08:24:39.00,13:55:29.92,PLC: VISION2 : 투입,08:31:21.43,21.39
4,20251015,Vision2,08:24:39.00,13:55:29.92,PLC: VISION2 : 투입,08:31:54.87,33.44
...,...,...,...,...,...,...,...
1301,20251016,Vision2,08:51:06.78,09:35:14.92,PLC: VISION2 : 투입,09:31:03.77,23.46
1302,20251016,Vision2,08:51:06.78,09:35:14.92,PLC: VISION2 : 투입,09:31:21.47,17.70
1303,20251016,Vision2,08:51:06.78,09:35:14.92,PLC: VISION2 : 투입,09:31:35.65,14.18
1304,20251010,Vision2,18:06:35.67,18:19:57.76,PLC: VISION2 : 투입,18:06:42.47,NaN


[Boxplot 대표값] final_ct = upper_outlier 제외 평균


,station,minimum,q1,median,q3,maximum,upper_outlier,final_ct
0,Vision1,11.04,15.14,18.95,23.77,36.56,36.71,19.34
1,Vision2,10.58,14.37,19.39,23.53,37.13,37.27,19.69


In [17]:
# =========================================================
# [수정] boxplot minimum 값이 발생한 구간(from_time~to_time) 찾기
#  - station 컬럼 중복 insert 방지
# =========================================================
def find_minimum_segments(ct_df: pd.DataFrame, station_name: str):
    cols = ["end_day", "station", "from_time", "to_time", "time", "ct"]

    if ct_df.empty or ct_df["ct"].dropna().empty:
        return pd.DataFrame(columns=[
            "station_name","minimum","lower_bound","upper_outlier",
            "end_day","station","from_time","to_time","time","ct"
        ])

    # ct 숫자 변환 안전 처리
    tmp = ct_df.copy()
    tmp["ct_num"] = pd.to_numeric(tmp["ct"], errors="coerce")
    x = tmp["ct_num"].dropna()

    if x.empty:
        return pd.DataFrame(columns=[
            "station_name","minimum","lower_bound","upper_outlier",
            "end_day","station","from_time","to_time","time","ct"
        ])

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr  # upper_outlier

    # whisker 범위 내 데이터
    in_mask = (tmp["ct_num"].notna()) & (tmp["ct_num"] >= lower) & (tmp["ct_num"] <= upper)
    in_df = tmp.loc[in_mask, cols + ["ct_num"]].copy()

    if in_df.empty:
        return pd.DataFrame(columns=[
            "station_name","minimum","lower_bound","upper_outlier",
            "end_day","station","from_time","to_time","time","ct"
        ])

    minimum_val = float(in_df["ct_num"].min())

    # minimum 발생 row들(복수 가능)
    hit = in_df[in_df["ct_num"] == minimum_val].copy()

    # 메타 컬럼 추가 (중복 방지: station은 station_name으로 별도 표기)
    hit.insert(0, "upper_outlier", round(float(upper), 2))
    hit.insert(0, "lower_bound", round(float(lower), 2))
    hit.insert(0, "minimum", round(minimum_val, 2))
    hit.insert(0, "station_name", station_name)

    # 보조 컬럼 제거
    hit = hit.drop(columns=["ct_num"])

    # 보기 좋게 정렬
    hit = hit.sort_values(["end_day", "from_time", "to_time", "time"]).reset_index(drop=True)
    return hit

min_seg_v1 = find_minimum_segments(df_c1_ct, "Vision1")
min_seg_v2 = find_minimum_segments(df_c2_ct, "Vision2")

print("====================================================")
print("[Vision1] boxplot minimum 값이 발생한 구간(from_time~to_time)")
print("====================================================")
display(min_seg_v1)

print("====================================================")
print("[Vision2] boxplot minimum 값이 발생한 구간(from_time~to_time)")
print("====================================================")
display(min_seg_v2)

[Vision1] boxplot minimum 값이 발생한 구간(from_time~to_time)


,station_name,minimum,lower_bound,upper_outlier,end_day,station,from_time,to_time,time,ct
0,Vision1,11.04,2.2,36.71,20251112,Vision1,08:34:14.93,17:43:11.56,10:07:04.69,11.04


[Vision2] boxplot minimum 값이 발생한 구간(from_time~to_time)


,station_name,minimum,lower_bound,upper_outlier,end_day,station,from_time,to_time,time,ct
0,Vision2,10.58,0.63,37.27,20251030,Vision2,20:25:58.41,22:24:34.27,21:18:55.10,10.58


In [18]:
# =========================================================
# 특정 interval(from_time ~ to_time)의 투입 CT DataFrame 생성
# (당신이 첨부한 형태 그대로)
# =========================================================

TARGET_END_DAY = "20251112"
STATION_NAME   = "Vision1"

FROM_TIME_STR = "08:34:14.93"
TO_TIME_STR   = "17:43:11.56"

INPUT_PHRASE  = "PLC: VISION1 : 투입"

# 문자열 → Timedelta
from_td = pd.to_timedelta(FROM_TIME_STR)
to_td   = pd.to_timedelta(TO_TIME_STR)

df_interval = df_c1[
    (df_c1["end_day"] == TARGET_END_DAY) &
    (df_c1["time_td"] >= from_td) &
    (df_c1["time_td"] <= to_td) &
    (df_c1["contents"].str.contains(INPUT_PHRASE, regex=False, na=False))
].copy()

df_interval = df_interval.sort_values("time_td").reset_index(drop=True)

# CT 계산 (현재 - 이전)
df_interval["ct"] = df_interval["time_td"].diff().dt.total_seconds().round(2)

# time 포맷
df_interval["time"] = df_interval["time_td"].apply(format_time)

# from_time / to_time 메타 컬럼 고정 삽입
df_interval.insert(1, "station", STATION_NAME)
df_interval.insert(2, "from_time", FROM_TIME_STR)
df_interval.insert(3, "to_time", TO_TIME_STR)

# 최종 컬럼 정렬 (첨부파일 형태)
df_interval = df_interval[
    ["end_day", "station", "from_time", "to_time", "contents", "time", "ct"]
]

display(df_interval)

,end_day,station,from_time,to_time,contents,time,ct
0,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,08:35:11.54,NaN
1,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,08:36:01.27,49.73
2,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,08:36:39.55,38.28
3,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,08:37:33.69,54.14
4,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,08:38:12.93,39.24
...,...,...,...,...,...,...,...
1341,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,17:25:41.59,14.83
1342,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,17:26:06.42,24.83
1343,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,17:26:22.19,15.77
1344,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,17:26:45.74,23.55


In [21]:
# =========================
# ct < 12초만 필터링
# =========================
df_interval_lt12 = df_interval[
    df_interval["ct"].notna() & (df_interval["ct"] < 13)
].reset_index(drop=True)

display(df_interval_lt12)

,end_day,station,from_time,to_time,contents,time,ct
0,20251112,Vision1,08:34:14.93,17:43:11.56,PLC: VISION1 : 투입,10:07:04.69,11.04
